# 01 — Self-Critique & Refinement

**Model:** Gemma 2 (9B) via Nebius AI Studio  
**Pattern:** Reflection Loop  
**Difficulty:** ⭐⭐☆☆☆

---

## The Problem

A vanilla LLM call is a one-shot process: you send a prompt, you get a response, done. The model has no mechanism to reconsider its own output.

For tasks where quality matters — writing, code review, analysis — a single pass is rarely enough. Humans revise. Good agents should too.

**Self-Critique** adds a second agent (or a second role for the same model) whose only job is to find flaws in the first output. A **Refinement** step then rewrites the original with those flaws corrected. This loop can run until the critique score crosses a threshold or a maximum iteration count is reached.

## Architecture

```
┌─────────────────────────────────────────────────────────┐
│                    LangGraph StateGraph                  │
│                                                          │
│   ┌──────────┐    ┌──────────┐    ┌───────────────┐     │
│   │  Draft   │───▶│ Critique │───▶│   Should we   │     │
│   │  Node    │    │  Node    │    │   stop?       │     │
│   └──────────┘    └──────────┘    └───────┬───────┘     │
│                        ▲                  │             │
│                        │           yes ───┘ no          │
│                        │                   │            │
│                   ┌────┴─────┐             ▼            │
│                   │  Refine  │◀────────────             │
│                   │  Node    │                          │
│                   └──────────┘                          │
└─────────────────────────────────────────────────────────┘
```

## When to Use This

- Content that will be read by humans (docs, emails, reports)
- Code generation where correctness is verifiable by a critic
- Any task where a quality rubric can be defined

**Tradeoff:** Each loop doubles your token cost. Set a sensible `max_iterations` (2–3 is usually enough).

## Setup

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_nebius import ChatNebius
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator

# Gemma 2 9B — good balance of speed and quality for iterative tasks
llm = ChatNebius(
    model="google/gemma-2-9b-it",
    temperature=0.4,
    api_key=os.environ["NEBIUS_API_KEY"]
)

print("LLM ready:", llm.model_name)

## State Definition

The graph passes a shared `State` dict between nodes. We track:
- `topic` — the original writing prompt
- `draft` — the current version of the text
- `critique` — the last critique produced
- `iteration` — how many refinement cycles we've completed
- `max_iterations` — the ceiling

In [ ]:
class State(TypedDict):
    topic: str
    draft: str
    critique: str
    iteration: int
    max_iterations: int

## Node Definitions

Each node is a plain Python function: `State → dict`. The returned dict is **merged** into the current state (not replaced).

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage


def draft_node(state: State) -> dict:
    """Produce the first draft of the content."""
    print(f"\n[Draft] Writing initial draft for: '{state['topic']}'")

    response = llm.invoke([
        SystemMessage(content=(
            "You are a skilled technical writer. "
            "Write clearly, concisely, and with concrete examples."
        )),
        HumanMessage(content=f"Write a short (150–200 word) explanation of: {state['topic']}")
    ])

    return {"draft": response.content, "iteration": 0}


def critique_node(state: State) -> dict:
    """Evaluate the current draft and return structured feedback."""
    print(f"\n[Critique] Iteration {state['iteration'] + 1} — reviewing draft...")

    response = llm.invoke([
        SystemMessage(content=(
            "You are a rigorous editor. Your job is to identify specific weaknesses in technical writing. "
            "Be direct. List at most 3 concrete issues: vague language, missing examples, poor structure, etc. "
            "If the writing is already strong, say 'APPROVED' and nothing else."
        )),
        HumanMessage(content=(
            f"Topic: {state['topic']}\n\n"
            f"Draft to review:\n{state['draft']}\n\n"
            "What specific improvements are needed?"
        ))
    ])

    return {"critique": response.content}


def refine_node(state: State) -> dict:
    """Rewrite the draft addressing all critique points."""
    print(f"\n[Refine] Applying critique to improve draft...")

    response = llm.invoke([
        SystemMessage(content=(
            "You are a technical writer performing a targeted revision. "
            "Address every critique point listed. Keep the length similar to the original."
        )),
        HumanMessage(content=(
            f"Topic: {state['topic']}\n\n"
            f"Current draft:\n{state['draft']}\n\n"
            f"Critique (fix ALL of these):\n{state['critique']}\n\n"
            "Write the improved version:"
        ))
    ])

    return {
        "draft": response.content,
        "iteration": state["iteration"] + 1
    }

## Routing Logic

After each critique, the router decides whether to refine again or stop. Two conditions trigger a stop:
1. The critic said `APPROVED` — the writing is good enough.
2. We've hit `max_iterations` — preventing infinite loops.

In [ ]:
def should_stop(state: State) -> str:
    """Route: 'stop' if approved or max iterations reached, else 'refine'."""
    if "APPROVED" in state["critique"].upper():
        print("\n[Router] ✅ Critique approved — stopping loop.")
        return "stop"

    if state["iteration"] >= state["max_iterations"]:
        print(f"\n[Router] ⏱️ Max iterations ({state['max_iterations']}) reached — stopping.")
        return "stop"

    print(f"\n[Router] 🔄 Iteration {state['iteration']} — refining...")
    return "refine"

## Build the Graph

In [ ]:
builder = StateGraph(State)

# Register nodes
builder.add_node("draft", draft_node)
builder.add_node("critique", critique_node)
builder.add_node("refine", refine_node)

# Entry point
builder.set_entry_point("draft")

# Edges
builder.add_edge("draft", "critique")
builder.add_conditional_edges(
    "critique",
    should_stop,
    {"stop": END, "refine": "refine"}
)
builder.add_edge("refine", "critique")  # loop back

graph = builder.compile()
print("Graph compiled successfully.")

## Live Demo

**Scenario:** We ask the agent to write a short explanation of "Why large language models hallucinate" — a nuanced topic where vague writing is easy to spot.

In [ ]:
initial_state = {
    "topic": "Why large language models hallucinate",
    "draft": "",
    "critique": "",
    "iteration": 0,
    "max_iterations": 3
}

final_state = graph.invoke(initial_state)

In [ ]:
print("=" * 60)
print(f"FINAL DRAFT (after {final_state['iteration']} refinement(s))")
print("=" * 60)
print(final_state["draft"])
print("\n" + "=" * 60)
print("LAST CRITIQUE")
print("=" * 60)
print(final_state["critique"])

## Key Takeaways

| Concept | Implementation |
|---------|---------------|
| **Reflection loop** | `critique → refine → critique` cycle in the graph |
| **Conditional stopping** | `should_stop()` checks approval keyword or iteration count |
| **Separation of concerns** | Writer and critic are distinct prompts — easy to tune independently |
| **Cost control** | `max_iterations` caps token spend without breaking the loop logic |

## What's Next

In **Notebook 02**, we add external tools to the agent — moving from pure text generation to grounded, factual output via web search and structured data retrieval.